# Vergleich von drei Pi0FromGoAT-Dateien

Dieses Notebook liest die Histogramme `missingEnergy`, `openingAngle`, `pi0Mass`, `missingMass` und `pi0ThetaCM` aus drei ROOT-Dateien und stellt jeweils alle drei Histogramme übereinander dar. Die ursprünglichen Bin-Inhalte werden nicht normiert.

In [1]:
import os
import sys
from pathlib import Path

# PyROOT aus der lokalen ROOT-Installation verfügbar machen.
root_prefix = Path("/opt/root_install")
os.environ["PATH"] = f"{root_prefix / 'bin'}:{os.environ.get('PATH', '')}"
sys.path.insert(0, str(root_prefix / "lib"))

import ROOT
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

ROOT.gROOT.SetBatch(True)
plt.rcParams.update({"figure.figsize": (10, 6.5), "axes.grid": True})
print(f"ROOT {ROOT.gROOT.GetVersion()}")

Welcome to JupyROOT 6.28/06
ROOT 6.28/06


## Eingabedateien

Hier die drei Dateipfade und die gewünschten Bezeichnungen in der Legende eintragen.

In [ ]:
inputs = [
    {"path": Path("/pfad/zur/datei1.root"), "label": "Datei 1", "color": "tab:blue"},
    {"path": Path("/pfad/zur/datei2.root"), "label": "Datei 2", "color": "tab:red"},
    {"path": Path("/pfad/zur/datei3.root"), "label": "Datei 3", "color": "tab:green"},
]

histogram_names = [
    "missingEnergy",
    "openingAngle",
    "pi0Mass",
    "missingMass",
    "pi0ThetaCM",
]

output_pdf = Path("Pi0FromGoAT_comparison_3files.pdf")
save_png = True

## Histogramme einlesen

Die Histogramme werden beim Einlesen in NumPy-Arrays kopiert. Die ROOT-Dateien können anschließend wieder geschlossen werden.

In [ ]:
def read_histogram(root_file, histogram_name):
    histogram = root_file.Get(histogram_name)
    if not histogram or not histogram.InheritsFrom("TH1"):
        raise KeyError(
            f"Histogramm '{histogram_name}' fehlt in '{root_file.GetName()}'."
        )

    axis = histogram.GetXaxis()
    number_of_bins = histogram.GetNbinsX()
    edges = np.array(
        [axis.GetBinLowEdge(i) for i in range(1, number_of_bins + 1)]
        + [axis.GetBinUpEdge(number_of_bins)],
        dtype=float,
    )
    values = np.array(
        [histogram.GetBinContent(i) for i in range(1, number_of_bins + 1)],
        dtype=float,
    )
    return {
        "edges": edges,
        "values": values,
        "x_title": axis.GetTitle(),
        "y_title": histogram.GetYaxis().GetTitle() or "Counts",
    }


histograms = {}
for entry in inputs:
    path = entry["path"].expanduser().resolve()
    root_file = ROOT.TFile.Open(str(path), "READ")
    if not root_file or root_file.IsZombie():
        raise OSError(f"ROOT-Datei konnte nicht geöffnet werden: {path}")

    histograms[entry["label"]] = {
        name: read_histogram(root_file, name) for name in histogram_names
    }
    root_file.Close()
    print(f"Eingelesen: {path}")

## Overlay-Plots erzeugen

In [ ]:
with PdfPages(output_pdf) as pdf:
    for histogram_name in histogram_names:
        figure, axis = plt.subplots()
        reference_edges = None

        for entry in inputs:
            data = histograms[entry["label"]][histogram_name]
            if reference_edges is None:
                reference_edges = data["edges"]
            elif not np.array_equal(reference_edges, data["edges"]):
                print(
                    f"Warnung: abweichendes Binning für {histogram_name} "
                    f"in {entry['label']}"
                )

            axis.stairs(
                data["values"],
                data["edges"],
                label=entry["label"],
                color=entry["color"],
                linewidth=1.7,
            )

        first_data = histograms[inputs[0]["label"]][histogram_name]
        axis.set_title(histogram_name)
        axis.set_xlabel(first_data["x_title"] or histogram_name)
        axis.set_ylabel(first_data["y_title"])
        axis.axhline(0.0, color="black", linewidth=0.7)
        axis.legend()
        figure.tight_layout()
        pdf.savefig(figure)
        if save_png:
            figure.savefig(f"{histogram_name}_comparison.png", dpi=160)
        plt.show()

print(f"PDF geschrieben: {output_pdf.resolve()}")